# 18 · Taxa de acerto na malha existente: protocolo principal, alarmes falsos, critérios e resolução

> **Regra da disciplina (aula 3):** especificação FOCUS → código gerado pelo assistente → executar → *gate* contra uma resposta já conhecida → aceitar.
> **Quando o gate falha, a restrição que faltou entra em `U`. O código não é remendado.**
>
> Cada passo abaixo tem três partes: a especificação FOCUS enviada ao assistente, o gate declarado **antes** do código (com a fonte da resposta conhecida e da tolerância) e o código que voltou, executado e gateado. O registro de uso de IA é exportado ao final (`registro_uso_ia.csv`).

**Pergunta do autor (16/11).** Como apurar, de forma estatisticamente defensável, a taxa de acerto do EAR calibrado na malha existente, se ela varia conforme o ano e a janela? E o que falta para aumentá-la, sem alarme falso?

**Protocolo principal (declarado antes dos resultados): origem móvel, para frente.** É o desenho de avaliação fora da amostra usado em previsão (Tashman, 2000) e a lógica dos testes de consistência entre períodos usados para comparar métodos de identificação de pontos críticos (Cheng e Washington, 2008). Para cada ano de origem T, o modelo é calibrado com os descarrilamentos de 2011 a T e conferido contra os descarrilamentos de T+1 a T+H. Horizonte principal H = 3 anos (origens de 2015 a 2022, oito conferências); sensibilidade com H = 1 e H = 5. O resultado é a média das origens, com a faixa e um intervalo de 95% por reamostragem de linhas inteiras.

**Duas medidas, porque respondem a perguntas diferentes.**

1. AUC: de cada 100 pares (um quilômetro com descarrilamento no período conferido, um sem), em quantos o quilômetro com descarrilamento recebeu a nota maior. Não depende de corte.
2. Taxa de acerto com corte de área (Chainey, Tompson e Uhlig, 2008): marcando os X% de quilômetros de maior nota, que percentual dos descarrilamentos do período conferido cai neles; e o índice de acurácia de previsão, que é essa taxa dividida por X. Reportada para X de 10, 20, 30, 40 e 50.

**Alarme falso.** Quilômetro marcado que não teve descarrilamento no horizonte. Como o evento é raro, a maior parte dos quilômetros marcados não terá evento em três anos, qualquer que seja o modelo; por isso o notebook reporta a concentração de risco (taxa nos marcados dividida pela taxa nos não marcados) e a repete só nos quilômetros que ainda não tinham descarrilamento até T.

**Métodos sob o mesmo protocolo.** Regra do EAR por faixas; Dummy da taxa da malha; critérios do EAR em medida contínua; EAR calibrado só com atributos da via; EAR calibrado mais histórico de acidentes da vizinhança, da linha e da malha (só possível em malha existente).

**Análises complementares, todas declaradas e todas reportadas.** (a) Grupos de critérios: o que cada grupo de atributos acrescenta aos critérios do EAR e o que se perde ao retirá-lo. (b) Resolução: a mesma conferência em blocos de 1, 2, 5 e 10 km, motivada pela incerteza de posição do RAAF (28% dos eventos em quilômetro inteiro). (c) Base americana (FRA): causas por tipo de via, para verificar se há critério não considerado.

**Gates.** 20.936 células; 2.681 descarrilamentos por via; janelas de calibração e de conferência disjuntas; oito origens no horizonte principal.

In [1]:
# === Setup: caminhos, bibliotecas e as três ferramentas (gate, gate_between, log_ia) ===
import os, sys, json, re, unicodedata, datetime as dt
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if os.environ.get("ET261400_BASE"):
    BASE = os.environ["ET261400_BASE"]                # definido pelo notebook 00 (execução em lote) ou pela execução local
elif IN_COLAB and os.path.exists("/content/ET261400/data"):
    BASE = "/content/ET261400"                        # pacote já descompactado na sessão do Colab
elif IN_COLAB and os.path.exists("/content/et261400_colab.zip"):
    import zipfile
    zipfile.ZipFile("/content/et261400_colab.zip").extractall("/content/ET261400")   # pacote enviado pelo painel de arquivos
    BASE = "/content/ET261400"
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE = "/content/drive/MyDrive/ET261400"          # pasta do projeto no Drive (ajustar se for outra)
else:
    BASE = os.environ.get("ET261400_BASE", os.path.abspath(".."))   # execução local: raiz do repositório
RAW, PROC = f"{BASE}/data/raw", f"{BASE}/data/processed"
FIG, TAB = f"{BASE}/results/figures", f"{BASE}/results/tables"
for d in (PROC, FIG, TAB):
    os.makedirs(d, exist_ok=True)
print("BASE =", BASE)

_GATES = []          # todo gate executado fica registrado aqui
_LOG_IA = []         # registro de uso de IA: passo, ferramenta, especificação, resultado

def gate(name, got, expected, tol=0.0, source=""):
    """Compara um número calculado com um número que já se tinha antes do código.
    tol vem da fonte (meio passo de arredondamento, faixa publicada), nunca é alargada até passar."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = abs(got - expected) <= tol
    _GATES.append(dict(gate=name, got=got, expected=expected, tol=tol, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, esperado {expected} (tol +/-{tol}) {('| ' + source) if source else ''}")
    return ok

def gate_between(name, got, low, high, source=""):
    """O mesmo, quando a fonte publica uma faixa e não um número."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = low <= got <= high
    _GATES.append(dict(gate=name, got=got, expected=f"[{low}, {high}]", tol=None, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, faixa [{low}, {high}] {('| ' + source) if source else ''}")
    return ok

def log_ia(passo, ferramenta, especificacao, resultado, verificacao):
    """Registro de uso de IA exigido pela disciplina: o que foi pedido, a que ferramenta, o que voltou e como foi verificado."""
    _LOG_IA.append(dict(data=dt.date.today().isoformat(), passo=passo, ferramenta=ferramenta,
                        especificacao=especificacao.strip(), resultado=resultado, verificacao=verificacao))

def exportar_registros(prefixo):
    pd.DataFrame(_GATES).to_csv(f"{TAB}/{prefixo}_gates.csv", index=False)
    pd.DataFrame(_LOG_IA).to_csv(f"{TAB}/{prefixo}_registro_uso_ia.csv", index=False)
    n_ok = sum(g["ok"] for g in _GATES)
    print(f"gates: {n_ok}/{len(_GATES)} PASS | registros de IA: {len(_LOG_IA)} | exportados em {TAB}")

def norm_txt(s):
    """Maiúsculas sem acento, espaços simples: chave de junção para nomes de pátios e estações."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).upper().strip()


BASE = /home/claude


In [2]:
import warnings; warnings.filterwarnings("ignore")
from scipy import sparse
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
SEED = 42
C0 = pd.read_csv(f"{PROC}/11_celulas_1km.csv")
C = C0[~C0["sentido"].astype(str).str.contains("Interromp")].sort_values(["ferrovia", "linha", "km_a"]).reset_index(drop=True); gate("células em operação", len(C), 20936, source="notebook 11")
E = pd.read_csv(f"{PROC}/11_eventos_celulas.csv"); E = E[E["cel_id"].isin(C["cel_id"])].copy(); E["ano"] = E["ano"].astype(int); EV = E[E["via"]]
gate("descarrilamentos por via nas células", len(EV), 2681, source="notebook 11")
idx = pd.Series(np.arange(len(C)), index=C["cel_id"]); N = len(C)
def matriz(ev):
    M_ = np.zeros((15, N))
    for k in range(15):
        s = ev[ev["ano"] == 2011 + k].groupby("cel_id").size(); M_[k, idx[s.index].values] = s.values
    return M_
CV_, CA_ = matriz(EV), matriz(E)
def cont(M_, a0, a1): return M_[a0 - 2011:a1 - 2011 + 1].sum(0)
GR = {"distância a pátio e AMV (EAR)": (["dmin_patio", "dmin_amv", "frac_zona_patio"] + [f"f16p{j}" for j in range(5)] + [f"f16a{j}" for j in range(5)], []),
      "raio de curva (EAR)": (["rmin_geo"] + [f"fr_lt{l}" for l in (300, 400, 500, 600, 800)], []),
      "superestrutura: trilho, dormente, fixação": (["taxa_dorm"], ["dormente", "perfil_trilho", "fixacao"]),
      "tráfego: capacidade vinculada, dias, linhas": (["cap_vinc", "dias_op"], ["sentido", "n_linhas"]),
      "carga por eixo e velocidade": (["carga_eixo_t", "vma_carregado"], []),
      "rampa e raio declarados": (["rampa_max_pct", "raio_min_m"], []),
      "sinalização e controle": ([], ["sinalizacao", "controle_circ"]),
      "bitola e gabarito": (["gab_vert_m", "gab_horiz_m"], ["bitola"])}
for num, cat in GR.values():
    for c in num: C[c] = pd.to_numeric(C[c], errors="coerce")
    for c in cat: C[c] = C[c].astype(str).str.strip().replace({"nan": "NA", "": "NA"})
def design(grupos):
    num = [c for g in grupos for c in GR[g][0]]; cat = [c for g in grupos for c in GR[g][1]]; X = C[num].copy(); med = X.median().fillna(0.0)
    for c in num: X[c + "_na"] = X[c].isna().astype(float); X[c] = X[c].fillna(med[c])
    X = pd.concat([X, pd.get_dummies(C[cat], dtype=float)], axis=1) if cat else X; return StandardScaler().fit_transform(X.values)
TODOS = list(GR); EARG = TODOS[:2]; X0 = design(TODOS); XE = design(EARG)
LN = C["len_km"].values; FERC = pd.factorize(C["ferrovia"])[0]; LIN = np.asarray(C["ferrovia"] + "|" + C["linha"].astype(str), dtype=object); LINC = pd.factorize(LIN)[0]; SEG = np.asarray(C["seg_id"].astype(str), dtype=object); KM = ((C["km_a"] + C["km_b"]) / 2).values
ri, ci = [], []
for l in range(LINC.max() + 1):
    i = np.where(LINC == l)[0]; k = KM[i]; M_ = (np.abs(k[:, None] - k[None, :]) <= 20) & ~np.eye(len(i), dtype=bool); a, b = np.where(M_); ri.append(i[a]); ci.append(i[b])
W = sparse.csr_matrix((np.ones(sum(len(a) for a in ri)), (np.concatenate(ri), np.concatenate(ci))), shape=(N, N)); nviz = np.asarray(W.sum(1)).ravel()
def historico(T):
    """Histórico com dados de 2011 até T, por ano de observação. Nada posterior a T."""
    n = T - 2010; hv = cont(CV_, 2011, T); ha = cont(CA_, 2011, T); b = (hv > 0).astype(float); d = pd.DataFrame({"b": b, "hv": hv})
    H_ = np.column_stack([hv / n, ha / n, b, (W @ b) / np.maximum(nviz, 1), (W @ hv) / np.maximum(nviz, 1) / n, d.groupby(SEG)["b"].transform("mean").values, d.groupby(LINC)["b"].transform("mean").values, d.groupby(FERC)["b"].transform("mean").values, d.groupby(FERC)["hv"].transform("mean").values / n])
    return H_, b
def cls16(d): return np.select([d <= 300, d <= 600, d <= 1200, d <= 1800], [4, 3, 2, 1], 0)
def rcls16(r_): return np.select([r_ < 500, r_ < 1200, r_ < 2300], [3, 2, 1], 0)
B2 = (0.5 * cls16(C["dmin_patio"].fillna(1e9)) / 4 + 0.5 * rcls16(C["rmin_geo"].fillna(1e6)) / 3).astype(float)
def auc(yy, s):
    n1 = yy.sum(); n0_ = len(yy) - n1; return float((rankdata(s)[yy == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0_)) if n1 >= 5 and n0_ >= 5 else np.nan
def auc_dentro(s, yy, grp=FERC):
    a, w = [], []
    for f in np.unique(grp):
        i = grp == f
        if yy[i].sum() >= 5 and yy[i].sum() < i.sum(): a.append(auc(yy[i], s[i])); w.append(yy[i].sum())
    return float(np.average(a, weights=w)) if a else np.nan
DES = np.random.default_rng(SEED).random(N)
def ordem(s): return np.lexsort((DES, -s))
def lr(Xtr, ytr, Xte): return LogisticRegression(C=0.5, max_iter=3000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
def notas(T, H):
    """Notas de cada método com dados até T; resposta conferida de T+1 a T+H."""
    assert T + H <= 2025; ycal = (cont(CV_, 2011, T) > 0).astype(int); cconf = cont(CV_, T + 1, T + H); yconf = (cconf > 0).astype(int)
    Htr, _ = historico(T - H); Hte, ja = historico(T); ytr = (cont(CV_, T - H + 1, T) > 0).astype(int); st = StandardScaler().fit(Htr)
    taxa = pd.Series(ycal).groupby(FERC).transform("mean").values
    S = {"Regra do EAR por faixas": B2, "Dummy: taxa da malha até T": taxa, "Critérios do EAR em medida contínua": lr(XE, ycal, XE), "EAR calibrado (atributos da via)": lr(X0, ycal, X0),
         "EAR calibrado + histórico de acidentes": lr(np.column_stack([X0, st.transform(Htr)]), ytr, np.column_stack([X0, st.transform(Hte)]))}
    return S, yconf, cconf, ja == 0
CORTES = (0.10, 0.20, 0.30, 0.40, 0.50)
def medidas(s, yconf, cconf, novo):
    o = ordem(s); cum = np.cumsum(LN[o]) / LN.sum(); d = dict(auc=auc(yconf, s), auc_dentro=auc_dentro(s, yconf), auc_km_sem_acidente_ate_T=auc(yconf[novo], s[novo]))
    for p in CORTES:
        m = np.zeros(N, bool); m[o[cum <= p]] = True; d[f"acerto_{int(p*100)}"] = float(cconf[m].sum() / cconf.sum()); d[f"km_atingidos_{int(p*100)}"] = float(yconf[m].sum() / yconf.sum())
        if p == 0.20:
            d["alarme_falso_20"] = float(1 - yconf[m].mean()); d["concentracao_20"] = float(yconf[m].mean() / yconf[~m].mean())
            a, b = yconf[m & novo].mean(), yconf[~m & novo].mean(); d["taxa_marcados_sem_acidente_ate_T"] = float(a); d["taxa_nao_marcados_sem_acidente_ate_T"] = float(b); d["concentracao_20_sem_acidente_ate_T"] = float(a / b)
    return d
log_ia("18.1 dados e funções", "Claude", "FOCUS: origem móvel para frente; AUC e taxa de acerto com corte de área; alarme falso e concentração de risco", "notas(T, H), medidas()", "2 gates de contagem")

[PASS] células em operação: obtido 20936, esperado 20936 (tol +/-0.0) | notebook 11
[PASS] descarrilamentos por via nas células: obtido 2681, esperado 2681 (tol +/-0.0) | notebook 11


In [3]:
# 1. Protocolo principal e sensibilidade ao horizonte
L, GUARDA = [], {}
for H in (3, 1, 5):
    for T in range(2015, 2026 - H):
        if T - H < 2012: continue
        S, yconf, cconf, novo = notas(T, H); GUARDA[(H, T)] = (S, yconf, cconf, novo)
        for nome, s in S.items(): L.append(dict(horizonte=H, origem=T, calibra=f"2011 a {T}", confere=f"{T+1} a {T+H}", metodo=nome, km_com_descarrilamento=int(yconf.sum()), deles_sem_acidente_ate_T=int(yconf[novo].sum()), **medidas(s, yconf, cconf, novo)))
    print("horizonte", H, "ok", flush=True)
R = pd.DataFrame(L); R.to_csv(f"{TAB}/18_origem_movel_por_origem.csv", index=False)
gate("origens no horizonte principal (3 anos)", int(R[R.horizonte == 3]["origem"].nunique()), 8, source="desenho: T de 2015 a 2022")
P = R[R.horizonte == 3]; COL = ["auc", "auc_dentro", "auc_km_sem_acidente_ate_T", "acerto_10", "acerto_20", "acerto_30", "acerto_40", "acerto_50", "alarme_falso_20", "concentracao_20", "concentracao_20_sem_acidente_ate_T"]
print("== PROTOCOLO PRINCIPAL: horizonte de 3 anos, 8 origens =="); p1 = P.groupby("metodo", sort=False)[COL].agg(["mean", "min", "max"]).round(3); display(p1.T)
print("== Por origem (AUC na malha inteira) =="); display(P.pivot(index="confere", columns="metodo", values="auc").round(3))
print("== Sensibilidade ao horizonte (média das origens) =="); display(R.groupby(["horizonte", "metodo"], sort=False)[["auc", "auc_dentro", "acerto_20", "acerto_40", "concentracao_20"]].mean().round(3))
# intervalo de 95% por reamostragem de linhas inteiras (notas fixas; 300 reamostragens)
rng = np.random.default_rng(SEED); cel_lin = [np.where(LINC == l)[0] for l in range(LINC.max() + 1)]; B = {m: [] for m in P.metodo.unique()}; Dif = []
for b in range(300):
    ii = np.concatenate([cel_lin[l] for l in rng.integers(0, len(cel_lin), len(cel_lin))]); v = {m: [] for m in B}
    for T in range(2015, 2023):
        S, yconf, _, _ = GUARDA[(3, T)]
        for m in B: v[m].append(auc(yconf[ii], S[m][ii]))
    for m in B: B[m].append(np.nanmean(v[m]))
    Dif.append(B["EAR calibrado (atributos da via)"][-1] - B["Regra do EAR por faixas"][-1])
ic = pd.DataFrame({m: dict(media=P[P.metodo == m]["auc"].mean(), ic95_inf=np.percentile(B[m], 2.5), ic95_sup=np.percentile(B[m], 97.5)) for m in B}).T.round(3); display(ic); ic.to_csv(f"{TAB}/18_ic95_auc.csv")
print("EAR calibrado menos regra: IC 95% de", round(np.percentile(Dif, 2.5), 3), "a", round(np.percentile(Dif, 97.5), 3))
p1.to_csv(f"{TAB}/18_protocolo_principal.csv")
log_ia("18.2 protocolo principal", "Claude", "FOCUS: 8 origens, horizonte de 3 anos; sensibilidade 1 e 5 anos; IC por reamostragem de linhas", "18_origem_movel_por_origem.csv, 18_protocolo_principal.csv, 18_ic95_auc.csv", "gate de número de origens")

horizonte 3 ok


horizonte 1 ok


horizonte 5 ok


[PASS] origens no horizonte principal (3 anos): obtido 8, esperado 8 (tol +/-0.0) | desenho: T de 2015 a 2022
== PROTOCOLO PRINCIPAL: horizonte de 3 anos, 8 origens ==


metodo                                   Regra do EAR por faixas  \
auc                                mean                    0.580   
                                   min                     0.565   
                                   max                     0.600   
auc_dentro                         mean                    0.577   
                                   min                     0.560   
                                   max                     0.597   
auc_km_sem_acidente_ate_T          mean                    0.577   
                                   min                     0.564   
                                   max                     0.589   
acerto_10                          mean                    0.169   
                                   min                     0.146   
                                   max                     0.213   
acerto_20                          mean                    0.263   
                                   min                     0.230   
                                   max                     0.322   
acerto_30                          mean                    0.401   
                                   min                     0.355   
                                   max                     0.463   
acerto_40                          mean                    0.507   
                                   min                     0.460   
                                   max                     0.560   
acerto_50                          mean                    0.627   
                                   min                     0.592   
                                   max                     0.669   
alarme_falso_20                    mean                    0.975   
                                   min                     0.963   
                                   max                     0.986   
concentracao_20                    mean                    1.449   
                                   min                     1.221   
                                   max                     1.863   
concentracao_20_sem_acidente_ate_T mean                    1.429   
                                   min                     1.243   
                                   max                     1.826   

metodo                                   Dummy: taxa da malha até T  \
auc                                mean                       0.688   
                                   min                        0.602   
                                   max                        0.741   
auc_dentro                         mean                       0.500   
                                   min                        0.500   
                                   max                        0.500   
auc_km_sem_acidente_ate_T          mean                       0.685   
                                   min                        0.602   
                                   max                        0.738   
acerto_10                          mean                       0.260   
                                   min                        0.157   
                                   max                        0.325   
acerto_20                          mean                       0.357   
                                   min                        0.267   
                                   max                        0.435   
acerto_30                          mean                       0.509   
                                   min                        0.374   
                                   max                        0.588   
acerto_40                          mean                       0.657   
                                   min                        0.442   
                                   max                        0.760   
acerto_50                          mean                       0.809   
                                   min                

== Por origem (AUC na malha inteira) ==


metodo,Critérios do EAR em medida contínua,Dummy: taxa da malha até T,EAR calibrado (atributos da via),EAR calibrado + histórico de acidentes,Regra do EAR por faixas
confere,,,,,
2016 a 2018,0.651,0.602,0.714,0.748,0.600
2017 a 2019,0.660,0.644,0.726,0.766,0.599
2018 a 2020,0.634,0.655,0.722,0.758,0.579
2019 a 2021,0.638,0.684,0.740,0.775,0.584
2020 a 2022,0.623,0.731,0.761,0.798,0.565
2021 a 2023,0.631,0.741,0.779,0.828,0.570
2022 a 2024,0.633,0.728,0.789,0.825,0.570
2023 a 2025,0.620,0.721,0.786,0.828,0.569


== Sensibilidade ao horizonte (média das origens) ==


auc  auc_dentro  \
horizonte metodo                                                      
3         Regra do EAR por faixas                 0.580       0.577   
          Dummy: taxa da malha até T              0.688       0.500   
          Critérios do EAR em medida contínua     0.636       0.646   
          EAR calibrado (atributos da via)        0.752       0.665   
          EAR calibrado + histórico de acidentes  0.791       0.706   
1         Regra do EAR por faixas                 0.581       0.576   
          Dummy: taxa da malha até T              0.686       0.500   
          Critérios do EAR em medida contínua     0.633       0.643   
          EAR calibrado (atributos da via)        0.755       0.673   
          EAR calibrado + histórico de acidentes  0.780       0.698   
5         Regra do EAR por faixas                 0.574       0.573   
          Dummy: taxa da malha até T              0.711       0.500   
          Critérios do EAR em medida contínua     0.630       0.645   
          EAR calibrado (atributos da via)        0.758       0.659   
          EAR calibrado + histórico de acidentes  0.794       0.710   

                                                  acerto_20  acerto_40  \
horizonte metodo                                                         
3         Regra do EAR por faixas                     0.263      0.507   
          Dummy: taxa da malha até T                  0.357      0.657   
          Critérios do EAR em medida contínua         0.356      0.587   
          EAR calibrado (atributos da via)            0.537      0.759   
          EAR calibrado + histórico de acidentes      0.604      0.821   
1         Regra do EAR por faixas                     0.272      0.512   
          Dummy: taxa da malha até T                  0.350      0.660   
          Critérios do EAR em medida contínua         0.361      0.585   
          EAR calibrado (atributos da via)            0.544      0.766   
          EAR calibrado + histórico de acidentes      0.595      0.800   
5         Regra do EAR por faixas                     0.245      0.486   
          Dummy: taxa da malha até T                  0.389      0.704   
          Critérios do EAR em medida contínua         0.342      0.569   
          EAR calibrado (atributos da via)            0.549      0.763   
          EAR calibrado + histórico de acidentes      0.605      0.840   

                                                  concentracao_20  
horizonte metodo                                                   
3         Regra do EAR por faixas                           1.449  
          Dummy: taxa da malha até T                        2.206  
          Critérios do EAR em medida contínua               2.219  
          EAR calibrado (atributos da via)                  4.622  
          EAR calibrado + histórico de acidentes            6.047  
1         Regra do EAR por faixas                           1.526  
          Dummy: taxa da malha até T                        2.203  
          Critérios do EAR em medida contínua               2.297  
          EAR calibrado (atributos da via)                  4.942  
          EAR calibrado + histórico de acidentes            6.135  
5         Regra do EAR por faixas                           1.341  
          Dummy: taxa da malha até T                        2.443  
          Critérios do EAR em medida contínua               2.090  
          EAR calibrado (atributos da via)                  4.650  
          EAR calibrado + histórico de acidentes            6.007

,media,ic95_inf,ic95_sup
Regra do EAR por faixas,0.580,0.546,0.623
Dummy: taxa da malha até T,0.688,0.594,0.762
Critérios do EAR em medida contínua,0.636,0.586,0.697
EAR calibrado (atributos da via),0.752,0.686,0.798
EAR calibrado + histórico de acidentes,0.791,0.738,0.826


EAR calibrado menos regra: IC 95% de 0.094 a 0.235


In [4]:
# 2. Grupos de critérios: o que cada um acrescenta aos critérios do EAR e o que se perde ao retirá-lo (horizonte de 3 anos, 8 origens)
def roda(grupos):
    Xg = design(grupos); a, d = [], []
    for T in range(2015, 2023):
        ycal = (cont(CV_, 2011, T) > 0).astype(int); yconf = (cont(CV_, T + 1, T + 3) > 0).astype(int); s = lr(Xg, ycal, Xg); a.append(auc(yconf, s)); d.append(auc_dentro(s, yconf))
    return float(np.mean(a)), float(np.mean(d))
L = [dict(configuracao="só critérios do EAR (pátio e AMV, raio)", **dict(zip(("auc", "auc_dentro"), roda(EARG)))), dict(configuracao="todos os grupos", **dict(zip(("auc", "auc_dentro"), roda(TODOS))))]
for g in TODOS[2:]: L.append(dict(configuracao="critérios do EAR + " + g, **dict(zip(("auc", "auc_dentro"), roda(EARG + [g])))))
for g in TODOS: L.append(dict(configuracao="todos menos " + g, **dict(zip(("auc", "auc_dentro"), roda([x for x in TODOS if x != g])))))
G = pd.DataFrame(L).round(3); G.to_csv(f"{TAB}/18_grupos_de_criterios.csv", index=False); display(G)
log_ia("18.3 grupos de critérios", "Claude", "FOCUS: acréscimo e retirada de um grupo por vez, mesmo protocolo", "18_grupos_de_criterios.csv", "comparação com a linha 'todos os grupos' da seção 1")

,configuracao,auc,auc_dentro
0,"só critérios do EAR (pátio e AMV, raio)",0.636,0.646
1,todos os grupos,0.752,0.665
2,"critérios do EAR + superestrutura: trilho, dor...",0.712,0.659
3,critérios do EAR + tráfego: capacidade vincula...,0.662,0.649
4,critérios do EAR + carga por eixo e velocidade,0.707,0.638
5,critérios do EAR + rampa e raio declarados,0.669,0.647
6,critérios do EAR + sinalização e controle,0.691,0.646
7,critérios do EAR + bitola e gabarito,0.739,0.660
8,todos menos distância a pátio e AMV (EAR),0.736,0.652
9,todos menos raio de curva (EAR),0.744,0.646


In [5]:
# 3. Resolução: a mesma conferência em blocos de 1, 2, 5 e 10 km (horizonte de 3 anos, 8 origens)
pos = pd.Series(KM).groupby(LINC).rank(method="first").values.astype(int) - 1; L = []
for r in (1, 2, 5, 10):
    blk = pd.factorize(pd.Series(LINC).astype(str) + "_" + pd.Series(pos // r).astype(str))[0]; fb = pd.Series(FERC).groupby(blk).first().values
    for T in range(2015, 2023):
        S, yconf, _, _ = GUARDA[(3, T)]; yb = pd.Series(yconf).groupby(blk).max().values
        for m, s in S.items():
            sb = pd.Series(s).groupby(blk).max().values if m.startswith(("Regra", "Dummy")) else 1 - np.exp(pd.Series(np.log1p(-np.clip(s, 0, 1 - 1e-9))).groupby(blk).sum().values)
            L.append(dict(bloco_km=r, origem=T, metodo=m, blocos=len(yb), blocos_com_descarrilamento=int(yb.sum()), auc=auc(yb, sb), auc_dentro=auc_dentro(sb, yb, fb)))
RS = pd.DataFrame(L); RS.to_csv(f"{TAB}/18_resolucao.csv", index=False); display(RS.groupby(["bloco_km", "metodo"], sort=False)[["blocos", "blocos_com_descarrilamento", "auc", "auc_dentro"]].mean().round(3))
log_ia("18.4 resolução", "Claude", "FOCUS: agregação das notas por bloco (probabilidade de ao menos um evento); regra e Dummy pelo máximo", "18_resolucao.csv", "bloco de 1 km reproduz a seção 1")

blocos  \
bloco_km metodo                                            
1        Regra do EAR por faixas                 20936.0   
         Dummy: taxa da malha até T              20936.0   
         Critérios do EAR em medida contínua     20936.0   
         EAR calibrado (atributos da via)        20936.0   
         EAR calibrado + histórico de acidentes  20936.0   
2        Regra do EAR por faixas                 10504.0   
         Dummy: taxa da malha até T              10504.0   
         Critérios do EAR em medida contínua     10504.0   
         EAR calibrado (atributos da via)        10504.0   
         EAR calibrado + histórico de acidentes  10504.0   
5        Regra do EAR por faixas                  4244.0   
         Dummy: taxa da malha até T               4244.0   
         Critérios do EAR em medida contínua      4244.0   
         EAR calibrado (atributos da via)         4244.0   
         EAR calibrado + histórico de acidentes   4244.0   
10       Regra do EAR por faixas                  2159.0   
         Dummy: taxa da malha até T               2159.0   
         Critérios do EAR em medida contínua      2159.0   
         EAR calibrado (atributos da via)         2159.0   
         EAR calibrado + histórico de acidentes   2159.0   

                                                 blocos_com_descarrilamento  \
bloco_km metodo                                                               
1        Regra do EAR por faixas                                    393.625   
         Dummy: taxa da malha até T                                 393.625   
         Critérios do EAR em medida contínua                        393.625   
         EAR calibrado (atributos da via)                           393.625   
         EAR calibrado + histórico de acidentes                     393.625   
2        Regra do EAR por faixas                                    374.125   
         Dummy: taxa da malha até T                                 374.125   
         Critérios do EAR em medida contínua                        374.125   
         EAR calibrado (atributos da via)                           374.125   
         EAR calibrado + histórico de acidentes                     374.125   
5        Regra do EAR por faixas                                    337.625   
         Dummy: taxa da malha até T                                 337.625   
         Critérios do EAR em medida contínua                        337.625   
         EAR calibrado (atributos da via)                           337.625   
         EAR calibrado + histórico de acidentes                     337.625   
10       Regra do EAR por faixas                                    296.125   
         Dummy: taxa da malha até T                                 296.125   
         Critérios do EAR em medida contínua                        296.125   
         EAR calibrado (atributos da via)                           296.125   
         EAR calibrado + histórico de acidentes                     296.125   

                                                   auc  auc_dentro  
bloco_km metodo                                                     
1        Regra do EAR por faixas                 0.580       0.577  
         Dummy: taxa da malha até T              0.688       0.500  
         Critérios do EAR em medida contínua     0.636       0.646  
         EAR calibrado (atributos da via)        0.752       0.665  
         EAR calibrado + histórico de acidentes  0.791       0.706  
2        Regra do EAR por faixas                 0.571       0.573  
         Dummy: taxa da malha até T              0.686       0.500  
         Critérios do EAR em medida contínua     0.640       0.650  
         EAR calibrado (atributos da via)        0.749       0.663  
         EAR calibrado + histórico de acidentes  0.788       0.703  
5        Regra do EAR por faixas                 0.551       0.566  
         Dummy: taxa da malha até T              0.689       0.500  
         Critérios do EA

In [6]:
# 4. Base americana (FRA, descarrilamentos por via de 2006 a 2026): causas por tipo de via, para checar critérios
F_ = pd.read_csv(f"{PROC}/fra_desc_via_2006_2026.csv"); print("narrativas:", len(F_))
t = pd.crosstab(F_["Track Type"].fillna("NA"), F_["familia"]); t["total"] = t.sum(1); t = t.sort_values("total", ascending=False); display(t); display((t.div(t["total"], axis=0) * 100).round(1))
display((F_["Primary Accident Cause"].value_counts().head(15) / len(F_) * 100).round(1).to_frame("% dos descarrilamentos por via"))
t.to_csv(f"{TAB}/18_fra_tipo_de_via_por_familia.csv"); F_["Primary Accident Cause"].value_counts().head(25).to_csv(f"{TAB}/18_fra_causas_mais_frequentes.csv")
log_ia("18.5 base americana", "Claude", "FOCUS: tabela descritiva; sem modelo", "18_fra_*.csv", "total de narrativas confere com o notebook 04")
exportar_registros("18")

narrativas: 11360


familia,F1 geometria e bitola,"F3 trilho, juntas e fixações",F4 AMV e aparelhos,"F5 plataforma, lastro e drenagem",F6 obras de arte e outros,total
Track Type,,,,,,
Yard,1830,2138,1421,155,123,5667
Main,1262,1388,311,264,62,3287
Industry,996,463,376,74,31,1940
Siding,204,137,72,23,3,439
NA,8,5,13,1,0,27


familia,F1 geometria e bitola,"F3 trilho, juntas e fixações",F4 AMV e aparelhos,"F5 plataforma, lastro e drenagem",F6 obras de arte e outros,total
Track Type,,,,,,
Yard,32.3,37.7,25.1,2.7,2.2,100.0
Main,38.4,42.2,9.5,8.0,1.9,100.0
Industry,51.3,23.9,19.4,3.8,1.6,100.0
Siding,46.5,31.2,16.4,5.2,0.7,100.0
NA,29.6,18.5,48.1,3.7,0.0,100.0


,% dos descarrilamentos por via
Primary Accident Cause,
Wide gage (due to defective or missing crossties),17.4
Broken Rail - Detail fracture from shelling or head check [Desc.Changed-Refer to Current/Prev.Guide,7.6
Switch point worn or broken,7.3
Broken Rail - Transverse/compound fissure [Desc.Changed-Refer to Current/Prev.Guides],6.1
Wide gage (due to defective or missing spikes or other rail fasteners),4.7
Broken Rail - Vertical split head [Desc.Changed-Refer to Current/Prev.Guides],4.2
Track alignment irregular (buckled/sunkink),3.7
Roadbed settled or soft,3.5
Broken Rail - Head and web separation (outside joint bar limits) [Desc.Changed-Refer to Current/Pre,3.5


gates: 3/3 PASS | registros de IA: 5 | exportados em /home/claude/results/tables


---
## Referências do protocolo (situação da verificação em 16/11/2026)

Chainey, S., Tompson, L., & Uhlig, S. (2008). The utility of hotspot mapping for predicting spatial patterns of crime. *Security Journal, 21*(1). Texto lido (definição da taxa de acerto e do índice de acurácia de previsão; desenho com várias janelas de passado e de futuro). Páginas: a confirmar.

Cheng, W., & Washington, S. (2008). New criteria for evaluating methods of identifying hot spots. *Transportation Research Record, 2083*, 76–85. https://doi.org/10.3141/2083-09. Resumo lido.

Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: an analysis and review. *International Journal of Forecasting, 16*(4), 437–450. Registro bibliográfico confirmado; resumo a ler antes de citar no artigo.

## Limites

Os atributos da via são os da Declaração de Rede de 2025 em todas as origens. As origens se sobrepõem no tempo e não são independentes; o intervalo por reamostragem de linhas trata a dependência espacial, não a temporal. O alarme falso em horizonte de três anos superestima o erro real, porque um quilômetro marcado pode vir a ter descarrilamento depois do horizonte.

**Declaração de uso de IA (regra da disciplina, três partes).** Ferramentas: Claude (Anthropic), usado pelo autor como assistente de programação a partir das especificações FOCUS registradas neste notebook, e Gemini do Colab quando indicado. Parte do trabalho: geração do código de cada passo e revisão do texto das células. Responsável pelo conteúdo: o autor, que definiu o problema, as fontes, as especificações, os gates e a verificação, e executou o notebook do zero antes de reportar qualquer número.